# CA Exam Checker AI — Evaluation Supervisor Fine-Tuning

This notebook prepares a separate **research-only evaluation-supervisor adapter** for both **CA Foundation and CA Intermediate**. Its job is to audit structured evaluation reports for internal consistency, correct checking mode, valid alternative-method credit, proportional partial credit, and consequential marking. It is **not yet a fully trained CA subject examiner** and does not overwrite the existing study-planner model.

**Course/material scope**
- Foundation: Accounting; Business Laws; Quantitative Aptitude; Business Economics.
- Intermediate: Advanced Accounting; Corporate and Other Laws; Taxation; Cost and Management Accounting; Auditing and Ethics; Financial Management and Strategic Management.
- Material target: exam-cycle-applicable MTP Series I & II, matching Suggested Answers/marking guidance where published, RTPs, previous-year question papers, and amendments/applicability notices. Keep course, paper, attempt, scheme and applicability version attached to every file.
- The first Colab run can train on synthetic examples while source links are catalogued. ICAI PDFs are **not downloaded by default**; written permission is required before storing them in a training/retrieval system. See the source-catalog section.

**Evidence limitations**
- Synthetic metrics only verify that the training pipeline runs; they do not establish academic marking accuracy.
- Downloaded question papers and Suggested Answers are **reference-only**, not supervised labels by themselves. Real supervised training needs human-verified student answers, score-by-component marks, reasons, selected checking mode and adjudicated outcomes, and permission to process the data.
- Keep this adapter advisory. Existing deterministic integrity gates and human review remain authoritative for uncertain cases.


In [ ]:
import os, sys, subprocess, json, torch
from pathlib import Path

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU runtime required. In Colab choose Runtime > Change runtime type > T4 GPU, then rerun this cell."
    )
print("✅ GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))

from google.colab import drive
drive.mount("/content/drive")


In [ ]:
REPO_DIR = "/content/AI-Exam"
BRANCH = "main"
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH,
         "https://github.com/thomasshelby-pixel/AI-Exam.git", REPO_DIR],
        check=True
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)

PROJECT_DIR = f"{REPO_DIR}/research/ca-evaluation-supervisor-ai"
%cd /content/AI-Exam/research/ca-evaluation-supervisor-ai
%pip install -q -r requirements-colab.txt
print("✅ Training sources ready:", PROJECT_DIR)


## 1. Official BoS catalogue — Foundation + Intermediate

This replaces one-by-one PDF uploads. The cell scans official ICAI/BoS pages and follows relevant official links to build a source catalogue for both courses. It prioritizes MTP/mock tests, suggested answers, RTP, previous-year papers, amendments and applicable study material. ICAI BoS announced September 2026 MTP Series I & II for **both Foundation and Intermediate**.

**Important rights gate:** public availability is not permission to store ICAI material in a retrieval system. The catalogue-only scan is safe to run now. PDF downloads are skipped unless you explicitly configure the written-permission acknowledgement inside the next cell. Nothing from a PDF is automatically placed into the training JSONL; reference PDFs are kept separate.


In [ ]:
# Official-link batch discovery + gated bulk download (no manual PDF uploads)
import csv, hashlib, json, re, time, urllib.parse
from datetime import datetime, timezone
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import fitz

UTC_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
DRIVE_ROOT = Path("/content/drive/MyDrive/CA_Exam_Checker_AI/training_runs")
CATALOG_DIR = DRIVE_ROOT / "bos_reference_catalog" / UTC_STAMP
CATALOG_DIR.mkdir(parents=True, exist_ok=False)
PDF_DIR = CATALOG_DIR / "reference_pdfs"
PDF_DIR.mkdir(parents=True, exist_ok=True)

# Official source pages only; discovery follows links staying on the allow-list.
ROOT_PAGES = [
    "https://boslive.icai.org/",
    "https://boslive.icai.org/bos_announcement.php",
    "https://www.icai.org/post/foundation-course",
    "https://www.icai.org/post/intermediate-course",
    "https://www.icai.org/post/suggested-answer-foundation-course",
    "https://www.icai.org/post/suggested-answer-intermediate-course",
]
ALLOWED_HOSTS = {"boslive.icai.org", "icai.org", "www.icai.org", "resource.cdn.icai.org"}
DISCOVERY_TERMS = (
    "mock test", "model test", "mtp", "suggested answer", "revision test", "rtp",
    "question paper", "previous year", "amendment", "applicability", "study material",
    "foundation", "intermediate", "september 2026", "may 2026", "january 2026"
)
COURSE_TERMS = {
    "CA_FOUNDATION": ("foundation",),
    "CA_INTERMEDIATE": ("intermediate",),
}
TYPE_TERMS = {
    "MTP": ("mock test", "model test", "mtp"),
    "SUGGESTED_ANSWER": ("suggested answer", "suggested answers"),
    "RTP": ("revision test", "rtp"),
    "PYQ": ("previous year", "question paper", "pyq"),
    "AMENDMENT": ("amendment", "applicability", "statutory update", "legislative update"),
    "STUDY_MATERIAL": ("study material", "module"),
}
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "Mozilla/5.0 (compatible; CAExamCheckerResearch/1.0; source-catalogue)"})

def allowed_url(url):
    try:
        u = urllib.parse.urlparse(url)
        return u.scheme == "https" and (u.hostname or "").lower() in ALLOWED_HOSTS
    except Exception:
        return False

def absolutize(base, href):
    url = urllib.parse.urljoin(base, (href or "").strip())
    return url.split("#", 1)[0] if allowed_url(url) else None

def classify(text):
    x = (text or "").lower()
    course = next((key for key, terms in COURSE_TERMS.items() if any(t in x for t in terms)), "UNKNOWN")
    kind = next((key for key, terms in TYPE_TERMS.items() if any(t in x for t in terms)), "OTHER")
    attempt = next((s for s in ("SEPTEMBER_2026", "MAY_2026", "JANUARY_2026", "SEPTEMBER_2025", "MAY_2025") if s.replace("_", " ").lower() in x), "ATTEMPT_UNCONFIRMED")
    series = next((s for s in ("SERIES_I", "SERIES_II", "SERIES_1", "SERIES_2") if s.replace("_", " ").lower() in x), "SERIES_UNCONFIRMED")
    return course, kind, attempt, series

page_queue = list(dict.fromkeys(ROOT_PAGES))
visited_pages, candidates = set(), {}
page_errors = []
# Bounded crawl: discover direct PDF links and a small number of relevant official detail pages.
for _depth in range(2):
    next_queue = []
    for page_url in page_queue[:30]:
        if page_url in visited_pages:
            continue
        visited_pages.add(page_url)
        try:
            response = SESSION.get(page_url, timeout=25, allow_redirects=True)
            final_url = response.url
            if not allowed_url(final_url):
                page_errors.append({"url": page_url, "error": "Redirected outside official allow-list; skipped."})
                continue
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")
            page_title = soup.title.get_text(" ", strip=True) if soup.title else ""
            for a in soup.find_all("a", href=True):
                href = absolutize(final_url, a.get("href"))
                if not href:
                    continue
                label = " ".join(a.get_text(" ", strip=True).split())
                context = " ".join([label, href, final_url])
                is_pdf = ".pdf" in urllib.parse.urlparse(href).path.lower() or "pdf" in href.lower()
                if is_pdf:
                    previous = candidates.get(href, {})
                    combined_label = " | ".join(x for x in [previous.get("link_text", ""), label] if x)
                    course, kind, attempt, series = classify(combined_label + " " + href + " " + final_url + " " + page_title)
                    candidates[href] = {
                        "url": href, "link_text": combined_label[:500], "discovered_on": final_url,
                        "course": course, "type": kind, "attempt": attempt, "series": series,
                    }
                elif _depth == 0 and any(term in context.lower() for term in DISCOVERY_TERMS):
                    if href not in visited_pages and href not in next_queue:
                        next_queue.append(href)
            time.sleep(0.15)
        except Exception as exc:
            page_errors.append({"url": page_url, "error": f"{type(exc).__name__}: {exc}"})
    page_queue = next_queue

candidate_rows = sorted(candidates.values(), key=lambda x: (x["course"], x["type"], x["attempt"], x["link_text"], x["url"]))
(CATALOG_DIR / "source_catalog.json").write_text(json.dumps(candidate_rows, indent=2, ensure_ascii=False), encoding="utf-8")
with (CATALOG_DIR / "source_catalog.csv").open("w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=["course", "type", "attempt", "series", "link_text", "url", "discovered_on"])
    w.writeheader()
    w.writerows(candidate_rows)
(CATALOG_DIR / "discovery_errors.json").write_text(json.dumps(page_errors, indent=2), encoding="utf-8")

print(f"✅ Scanned {len(visited_pages)} official pages; found {len(candidate_rows)} PDF/link candidates.")
print("Catalog saved:", CATALOG_DIR / "source_catalog.csv")
print("PDF downloads are OFF by default. First review the catalogue:")
for idx, row in enumerate(candidate_rows, start=1):
    print(f'{idx:03d} | {row["course"]:<17} | {row["type"]:<16} | {row["attempt"]:<22} | {row["link_text"][:90]} | {row["url"]}')
if page_errors:
    print(f"\n⚠️ {len(page_errors)} pages could not be read (the BoS site may block automated requests). See discovery_errors.json.")

# Rights gate: leave False until written permission specifically covers storing/retrieving these PDFs.
ALLOW_ICAi_PDF_DOWNLOADS = False
WRITTEN_PERMISSION_ACK = ""
REQUIRED_ACK = "I_HAVE_WRITTEN_PERMISSION_FROM_ICAI_FOR_AI_TRAINING_OR_RETRIEVAL"
MAX_FILES = 100
MAX_FILE_BYTES = 80 * 1024 * 1024
MAX_TOTAL_BYTES = 2 * 1024 * 1024 * 1024

if not ALLOW_ICAi_PDF_DOWNLOADS or WRITTEN_PERMISSION_ACK != REQUIRED_ACK:
    print("\n⏸️ Skipping all PDF downloads. Catalogue-only mode completed.")
    print("Only enable download after obtaining written permission; changing a flag is not a substitute for permission.")
else:
    selected = [
        row for row in candidate_rows
        if row["course"] in {"CA_FOUNDATION", "CA_INTERMEDIATE"}
        and row["type"] in {"MTP", "SUGGESTED_ANSWER", "RTP", "PYQ", "AMENDMENT", "STUDY_MATERIAL"}
        and row["attempt"] in {"SEPTEMBER_2026", "MAY_2026", "JANUARY_2026", "SEPTEMBER_2025", "MAY_2025", "ATTEMPT_UNCONFIRMED"}
    ]
    # Keep an audit trail and enforce conservative size/count limits to avoid accidental bulk pulls.
    selected = selected[:MAX_FILES]
    total_bytes, file_records = 0, []
    for idx, item in enumerate(selected, start=1):
        url = item["url"]
        rec = {**item, "download_status": "PENDING"}
        try:
            if not allowed_url(url):
                raise ValueError("Source URL not on official allow-list")
            with SESSION.get(url, timeout=45, stream=True, allow_redirects=True) as resp:
                if not allowed_url(resp.url):
                    raise ValueError("PDF redirected outside official allow-list")
                resp.raise_for_status()
                declared = int(resp.headers.get("content-length") or 0)
                if declared and declared > MAX_FILE_BYTES:
                    raise ValueError(f"File exceeds per-file cap ({declared} bytes)")
                chunks, n = [], 0
                for chunk in resp.iter_content(chunk_size=1024 * 1024):
                    if not chunk:
                        continue
                    n += len(chunk)
                    if n > MAX_FILE_BYTES or total_bytes + n > MAX_TOTAL_BYTES:
                        raise ValueError("Download size limit reached")
                    chunks.append(chunk)
                raw = b"".join(chunks)
            if not raw.startswith(b"%PDF"):
                raise ValueError("Response was not a PDF file")
            file_name = f'{idx:03d}_{item["course"].lower()}_{item["type"].lower()}_{hashlib.sha256(url.encode()).hexdigest()[:10]}.pdf'
            path = PDF_DIR / file_name
            path.write_bytes(raw)
            total_bytes += len(raw)
            doc = fitz.open(stream=raw, filetype="pdf")
            text = "\n\n".join(f"--- Page {p.number+1} ---\n{p.get_text('text')}" for p in doc).strip()
            text_path = path.with_suffix(".txt")
            if len(text) >= 100:
                text_path.write_text(text, encoding="utf-8")
                extraction_status = "TEXT_EXTRACTED"
            else:
                extraction_status = "OCR_REQUIRED_OR_EMPTY_TEXT"
            rec.update({
                "download_status": "DOWNLOADED",
                "saved_pdf": str(path),
                "sha256": hashlib.sha256(raw).hexdigest(),
                "bytes": len(raw),
                "page_count": len(doc),
                "extracted_text_path": str(text_path) if extraction_status == "TEXT_EXTRACTED" else None,
                "extracted_characters": len(text),
                "extraction_status": extraction_status,
                "training_status": "REFERENCE_ONLY_NOT_A_SUPERVISED_LABEL",
            })
        except Exception as exc:
            rec.update({"download_status": "SKIPPED_OR_FAILED", "error": f"{type(exc).__name__}: {exc}"})
        file_records.append(rec)
        print(f'{idx}/{len(selected)} {rec["download_status"]}: {item["course"]} | {item["type"]} | {item["link_text"][:70]}')
        time.sleep(0.15)
    manifest = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "scope": ["CA_FOUNDATION", "CA_INTERMEDIATE"],
        "source_policy": "Official ICAI/BoS only; downloaded only after explicit written-permission acknowledgement.",
        "reference_files_only": True,
        "total_downloaded_bytes": total_bytes,
        "files": file_records,
    }
    (CATALOG_DIR / "reference_manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("\n✅ Batch import finished. No PDF text was added to train/validation/test JSONL.")
    print(json.dumps({"catalog_dir": str(CATALOG_DIR), "downloaded_files": sum(r["download_status"] == "DOWNLOADED" for r in file_records), "bytes": total_bytes}, indent=2))


In [ ]:
# Generate reproducible synthetic examples for train / validation / held-out test.
# The generator does not read Google Drive, student uploads, or ICAI PDFs.
subprocess.run(
    [sys.executable, "generate_supervisor_dataset.py",
     "--output-dir", "/content/ca-evaluation-supervisor-data",
     "--train-per-category", "160",
     "--validation-per-category", "30",
     "--test-per-category", "40",
     "--seed", "20261017"],
    check=True
)
manifest = json.loads(Path("/content/ca-evaluation-supervisor-data/dataset_manifest.json").read_text())
print(json.dumps(manifest, indent=2))
assert manifest["dataset_kind"] == "synthetic_only"
assert manifest["counts"]["train"]["examples"] > manifest["counts"]["test"]["examples"]


## Train an isolated QLoRA adapter

The trainer deliberately refuses CPU-only runtime and refuses output locations that could overwrite the old planner model. Default training uses a T4 GPU, 4-bit QLoRA, and a new versioned Drive output folder.

Do not rerun the training cell while it is running. A completed run should save adapter weights plus a training manifest.


In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/CA_Exam_Checker_AI/training_runs/evaluation_supervisor_smollm2_360m_v1_20261017"
subprocess.run(
    [sys.executable, "train_supervisor_qlora.py",
     "--train-file", "/content/ca-evaluation-supervisor-data/train.jsonl",
     "--validation-file", "/content/ca-evaluation-supervisor-data/validation.jsonl",
     "--output-dir", OUTPUT_DIR,
     "--base-model", "HuggingFaceTB/SmolLM2-360M-Instruct",
     "--epochs", "3",
     "--batch-size", "2",
     "--grad-accum", "8",
     "--learning-rate", "0.0001",
     "--max-seq-length", "1536"],
    check=True
)
print("✅ Training process exited successfully:", OUTPUT_DIR)


## Evaluate on held-out synthetic cases

This runs the fine-tuned adapter on held-out examples and (for comparison) the untouched base model. The report includes JSON parse rate, decision accuracy, issue-code precision/recall/F1, exact issue-set accuracy, and the rate at which flagged synthetic cases were incorrectly passed.

**Do not describe these synthetic metrics as ICAI grading accuracy.**


In [ ]:
EVAL_DIR = "/content/drive/MyDrive/CA_Exam_Checker_AI/training_runs/evaluation_supervisor_evaluation_v1_20261017"
subprocess.run(
    [sys.executable, "evaluate_supervisor.py",
     "--test-file", "/content/ca-evaluation-supervisor-data/test.jsonl",
     "--adapter-dir", OUTPUT_DIR,
     "--base-model", "HuggingFaceTB/SmolLM2-360M-Instruct",
     "--output-dir", EVAL_DIR,
     "--limit", "120",
     "--compare-base"],
    check=True
)
metrics_path = Path(EVAL_DIR) / "metrics.json"
metrics = json.loads(metrics_path.read_text())
print(json.dumps(metrics, indent=2))


In [ ]:
# Package only the root adapter/tokenizer files and evaluation evidence.
# Trainer checkpoint directories are intentionally excluded from the deployment package.
import shutil, zipfile

EXPORT_DIR = "/content/drive/MyDrive/CA_Exam_Checker_AI/training_runs/evaluation_supervisor_smollm2_360m_v1_20261017_EXPORT"
Path(EXPORT_DIR).mkdir(parents=True, exist_ok=True)
adapter_root = Path(OUTPUT_DIR)
for file_path in adapter_root.iterdir():
    if file_path.is_file():
        shutil.copy2(file_path, Path(EXPORT_DIR) / file_path.name)
for name in ["metrics.json", "base_metrics.json", "predictions.jsonl", "base_predictions.jsonl"]:
    source = Path(EVAL_DIR) / name
    if source.is_file():
        shutil.copy2(source, Path(EXPORT_DIR) / name)

package_path = "/content/drive/MyDrive/CA_Exam_Checker_AI/training_runs/evaluation_supervisor_adapter_package_20261017.zip"
with zipfile.ZipFile(package_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for file_path in Path(EXPORT_DIR).iterdir():
        if file_path.is_file():
            archive.write(file_path, arcname=file_path.name)

print("✅ Adapter package:", package_path)
print("✅ Evaluation evidence:", str(Path(EXPORT_DIR) / "metrics.json"))
print("⚠️ This is a research artifact; it is not wired into the live website yet.")


## Deployment gate — intentionally not automatic

The model has **not** been deployed by running this notebook. Next, review:
1. base vs fine-tuned synthetic metrics;
2. whether the adapter returns valid JSON and avoids false PASS decisions;
3. a separate benchmark of human-reviewed, legally permitted real evaluation examples.

Only then should we build an inference endpoint and connect it to the website. The model must never override the deterministic integrity gates or approve a copy when references, answer mapping, or handwriting are unresolved.
